# Parsimony: LoRA fine-tune of a small model
Run on Colab: **Runtime > Change runtime type > T4 GPU**. Run cells top to bottom.
Before starting, upload `ml.zip` (the repo's `ml/` folder zipped) using the Files panel on the left.

What happens: (1) base model is scored on the 100 hand-labeled test cases *untouched*, (2) it is trained
on 3,000 synthetic examples with LoRA, (3) the tuned model is scored on the same 100 cases.
The test cases are never used for training.

In [ ]:
!pip -q install -U transformers peft trl datasets accelerate
!pip -q uninstall -y torchao   # Colab ships an old torchao that newer peft rejects; we do not use it
!unzip -oq ml.zip
import sys; sys.path.insert(0, 'ml'); sys.path.insert(0, 'ml/eval')
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
import json, time, datetime as dt, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from prompt import make_prompt
import score

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'   # small, Apache-2.0, knows some Korean; change here to try another
tok = AutoTokenizer.from_pretrained(BASE)
cases = score.load_cases()

def run_model(model, name):
    model.eval(); preds = []; secs = []
    for c in cases:
        p = make_prompt(c['text'], dt.date.fromisoformat(c['reference_date']), c['default_currency'])
        ids = tok(p, return_tensors='pt').to(model.device)
        t = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=220, do_sample=False, pad_token_id=tok.eos_token_id)
        secs.append(time.perf_counter() - t)
        text = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        try: obj = json.loads(text)
        except Exception: obj = None
        preds.append(obj)
    print(f'== {name} ==  median {sorted(secs)[len(secs)//2]:.2f}s per answer (T4 GPU)')
    print(score.report(cases, preds))
    return preds

## 1. Baseline: the base model with no training

In [ ]:
RUN_BASE = False   # the untuned model scored 0/100 in round 1; set True to measure it again (~10 min)
if RUN_BASE:
    base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
    base_preds = run_model(base, 'base model, untuned')


## 2. Train (LoRA)
LoRA freezes the model and trains small add-on matrices (well under 2% of the weights).

In [ ]:
import os
TRAIN = 'ml/data/train_v2.jsonl' if os.path.exists('ml/data/train_v2.jsonl') else 'ml/data/train.jsonl'
print('training on', TRAIN)
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files=TRAIN)['train'].shuffle(seed=1)
ds = ds.map(lambda r: {'prompt': r['prompt'], 'completion': ' ' + r['completion'] + tok.eos_token},
            remove_columns=['lang'])
split = ds.train_test_split(test_size=0.05, seed=1)   # 5% only to watch the loss; NOT the test set

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg = SFTConfig(output_dir='out', num_train_epochs=2, per_device_train_batch_size=8,
                gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type='cosine',
                warmup_steps=20, logging_steps=20, eval_strategy='epoch', save_strategy='no',
                fp16=True, max_length=512, report_to='none')
# Free the GPU first: the untuned model from step 1 is still loaded and would crowd this one out.
import gc
for _n in ('base', 'model', 'trainer'):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
print(f'GPU memory in use before training: {torch.cuda.memory_allocated() / 1e9:.1f} GB (should be ~0)')
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32, device_map={'': 0})  # all on the GPU
trainer = SFTTrainer(model=model, args=cfg, train_dataset=split['train'], eval_dataset=split['test'],
                     processing_class=tok, peft_config=lora)
trainer.train()
trainer.model.save_pretrained('parsimony-lora')   # tiny adapter folder: download it from the Files panel

## 3. Evaluate the tuned model on the 100 held-out cases

In [ ]:
trainer.model.half()
tuned_preds = run_model(trainer.model, 'fine-tuned (LoRA)')
rows = [{'text': c['text'], 'output': o} for c, o in zip(cases, tuned_preds)]
open('preds_tuned.jsonl','w').write('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows))

## 4. Read the result honestly
Copy the two reports (base and tuned) and send them back. Compare with the Gemini number from
`ml/eval/run_gemini.py`. Whatever the numbers are, those are the numbers that go in the README.